# 6. Generated words for the word recogniser (E6)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chingkheinganba231005/Meitei-Mayek-Diffusion-model/blob/main/notebooks/6_generated_words_for_recognition.ipynb)

**Runtime: A100.** About 1 hour. Run after notebooks 3 to 5, if they finished within their budget.

1. **Generate** 20,000 training-lexicon words (texts drawn as the synthesiser draws them,
   seed 300) in the author's hand, few-shot from the 4 fixed references.
2. **Train the deployed recogniser further**, twice, 5,000 steps of 64 words each (lr 1e-4
   after 200 warm-up steps): (i) on the synthesiser's words only, as its training renders them;
   (ii) on half synthesiser and half generated words.
3. **Score** the starting model and both copies on the 96 real words that are not references,
   and on the 5,000 synthetic test words, to check that nothing is forgotten; with bootstrap
   intervals, since the real set is small.

The real words' sign positions informed the recogniser's latest synthesiser setting (signs above
also written beside their letter); the paper says so.

Inputs: `WORK/runs/main/final.pt`, `results/eval_val_grid.json`, `WORK/data/owner` (notebook 4),
and from the word-recognition project `glyphs/train.npz`, `lexicon/train.tsv`, `synth/test.tar`.
Outputs: `WORK/e6/`, `WORK/runs/e6_*.pt`, `results/e6_recognition.json`.

In [ ]:
import os

# Where things are. Change these to match your Drive.
WORK = "/content/drive/MyDrive/meitei-word-diffusion"         # this project's working folder
WORK_OLD = "/content/drive/MyDrive/meitei-word-recognition"   # the word-recognition project's folder
REPO = "chingkheinganba231005/Meitei-Mayek-Diffusion-model"
BRANCH = "main"
WORD_REPO = "chingkheinganba231005/meitei-mayek-word-recognition"
WORD_COMMIT = "bdee9e7"                                       # the synthesiser this project builds on
CONTENT = "/content"                                          # the runtime's own disk

HF_REC = "Chingkheinganba/handwritten-meitei-mayek-word-recognition"   # the deployed word recogniser
REC_REVISION = "main"                       # its version is recorded in results/recogniser_source.json
HWD_COMMIT = "eabb5b5"                      # the HWD package (github.com/aimagelab/HWD)

N_WORDS = 20000
TEXT_SEED = 300
TRAIN_STEPS = 5000
WORKERS = max((os.cpu_count() or 4) - 2, 2)   # processes rendering synthetic words

DISCONNECT_AT_END = True    # the last cell disconnects the runtime (Colab bills a connected runtime)

In [ ]:
import json, re, shutil, subprocess, sys, time
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
os.chdir(CONTENT)
CODE, WORD = f"{CONTENT}/code", f"{CONTENT}/word_repo"


def run(*args):
    """Runs a command, shows its output, and stops the notebook if it fails."""
    r = subprocess.run([str(a) for a in args], capture_output=True, text=True)
    if (r.stdout + r.stderr).strip():
        print(r.stdout + r.stderr)
    if r.returncode:
        raise SystemExit(f"exit code {r.returncode}: {' '.join(str(a) for a in args)[:200]}")


def stream(*args):
    """Runs a long command, showing its output as it comes; stops the notebook if it fails.
    Returns the output."""
    p = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        print(line, end="")
        lines.append(line)
    if p.wait():
        raise SystemExit(f"exit code {p.returncode}: {' '.join(str(a) for a in args)[:200]}")
    return "".join(lines)


def clone(url, path, ref):
    """A fresh copy of a repository at a branch or a commit."""
    shutil.rmtree(path, ignore_errors=True)
    run("git", "clone", "-q", url, path)
    run("git", "-C", path, "checkout", "-q", ref)


clone(f"https://github.com/{REPO}.git", CODE, BRANCH)
clone(f"https://github.com/{WORD_REPO}.git", WORD, WORD_COMMIT)
sys.path[:0] = [CODE, WORD]
os.chdir(CODE)

run(sys.executable, "-m", "pip", "install", "-q", "diffusers==0.41.0", "safetensors", "huggingface_hub", "timm")

for d in ("data", "runs", "results", "eval"):
    os.makedirs(f"{WORK}/{d}", exist_ok=True)
run("git", "-C", CODE, "log", "-1", "--format=code: %h %s")

run("nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader")

REC = f"{CONTENT}/recogniser"


def get_recogniser():
    """The deployed recogniser (recogniser.pt, char_lm.pkl, decoding.json) at the version recorded the
    first time (results/recogniser_source.json), so that every notebook scores with the same one."""
    from huggingface_hub import HfApi, snapshot_download
    kept = f"{WORK}/results/recogniser_source.json"
    if os.path.exists(kept):
        source = json.loads(Path(kept).read_text())
    else:
        source = {"repo": HF_REC, "revision": REC_REVISION, "sha": HfApi().model_info(HF_REC, revision=REC_REVISION).sha}
    if not os.path.exists(f"{REC}/source.json"):
        snapshot_download(HF_REC, revision=source["sha"], local_dir=REC,
                          allow_patterns=["recogniser.pt", "char_lm.pkl", "decoding.json", "results.json"])
        Path(f"{REC}/source.json").write_text(json.dumps(source))
    if not os.path.exists(kept):
        shutil.copy(f"{REC}/source.json", kept)
    print("recogniser:", json.loads(Path(f"{REC}/source.json").read_text()))


get_recogniser()

WEIGHTS, GRID, OWNER = f"{WORK}/runs/main/final.pt", f"{WORK}/results/eval_val_grid.json", f"{WORK}/data/owner"
for f in (WEIGHTS, f"{OWNER}/adapt/info.json", f"{WORK}/results/owner_split.json"):
    if not os.path.exists(f):
        raise SystemExit(f"{f} does not exist yet: run notebooks 2 to 4 first")
INPUTS = f"{CONTENT}/inputs"
for f in ("glyphs/train.npz", "lexicon/train.tsv", "synth/test.tar"):
    os.makedirs(os.path.dirname(f"{INPUTS}/{f}"), exist_ok=True)
    if not os.path.exists(f"{INPUTS}/{f}"):
        shutil.copy(f"{WORK_OLD}/{f}", f"{INPUTS}/{f}")
E6 = f"{WORK}/e6"
os.makedirs(E6, exist_ok=True)

## 1. Generate

In [ ]:
GENERATED = f"{CONTENT}/e6/generated.tar"
os.makedirs(f"{CONTENT}/e6", exist_ok=True)
if not os.path.exists(f"{E6}/generated.tar"):
    stream(sys.executable, "-u", "scripts/recognition.py", "generate", "--word-repo", WORD, "--weights", WEIGHTS,
           "--setting", GRID, "--owner", OWNER, "--lexicon", f"{INPUTS}/lexicon/train.tsv", "--n", N_WORDS,
           "--seed", TEXT_SEED, "--out", GENERATED)
    shutil.copy(GENERATED, f"{E6}/generated.tar")
    shutil.copy(GENERATED + ".json", f"{E6}/generated.tar.json")
elif not os.path.exists(GENERATED):
    shutil.copy(f"{E6}/generated.tar", GENERATED)

## 2. The recogniser trained further

In [ ]:
for name, extra in (("synth", []), ("mixed", ["--generated", GENERATED])):
    out = f"{WORK}/runs/e6_{name}.pt"
    if not os.path.exists(out):
        stream(sys.executable, "-u", "scripts/recognition.py", "finetune", "--word-repo", WORD, "--recogniser", REC,
               "--glyphs", f"{INPUTS}/glyphs/train.npz", "--lexicon", f"{INPUTS}/lexicon/train.tsv",
               "--train-steps", TRAIN_STEPS, "--workers", WORKERS, "--out", out, *extra)

## 3. Score

In [ ]:
stream(sys.executable, "-u", "scripts/recognition.py", "score", "--word-repo", WORD, "--recogniser", REC,
       "--models", f"start={REC}/recogniser.pt", f"synth={WORK}/runs/e6_synth.pt", f"mixed={WORK}/runs/e6_mixed.pt",
       "--real", f"{WORD}/real_words", "--split", f"{WORK}/results/owner_split.json",
       "--synth-test", f"{INPUTS}/synth/test.tar", "--work", f"{CONTENT}/e6/score",
       "--out", f"{WORK}/results/e6_recognition.json")

## Disconnect

Colab bills a connected runtime whether or not it is working. The cell below disconnects it
(`DISCONNECT_AT_END = True`); otherwise use *Runtime > Disconnect and delete runtime*. Every
result is already on Drive.

In [ ]:
# Colab bills a connected runtime whether or not it is working: disconnect it now.
if DISCONNECT_AT_END:
    from google.colab import runtime
    runtime.unassign()
else:
    print("Finished. Disconnect the runtime: Runtime > Disconnect and delete runtime.")